In [ ]:
import re
import os
import warnings
from transformers import pipeline

# 1. SILENCIAR AVISOS
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
warnings.filterwarnings("ignore")

# 2. CARGA DEL MODELO
print("⏳ Cargando cerebro de IA...")
clasificador = pipeline(
    "zero-shot-classification", 
    model="Recognai/bert-base-spanish-wwm-cased-xnli",
    framework="pt",
    device=-1
)

# 3. ESCUDO DE SEGURIDAD (PII)
def analizar_seguridad(texto):
    patrones = {
        "DNI": r"\b\d{8}[A-Z]\b",
        "Teléfono": r"\b[6789]\d{8}\b",
        "Matrícula": r"\b\d{4}[A-Z]{3}\b",
        "IBAN": r"\bES\d{22}\b",
        "Póliza": r"\b\d{8}\b" 
    }
    hallazgos = []
    for tipo, regex in patrones.items():
        matches = re.findall(regex, texto, re.IGNORECASE)
        for m in matches:
            if tipo == "Póliza": continue 
            anonimo = m[:2] + "*" * (len(m)-4) + m[-2:]
            hallazgos.append({"tipo": tipo, "anonimo": anonimo})
    return hallazgos

# 4. CONFIGURACIÓN DE ETIQUETAS Y UMBRALES
etiquetas = [
    "accidente de tráfico, choque o colisión con otro vehículo",
    "asistencia en carretera, avería mecánica o necesidad de grúa",
    "daños por agua, goteras, filtraciones o rotura de tuberías",
    "incendio, robo o daños por vandalismo",
    "consulta de recibos pagados, pendientes o próximos cobros",
    "modificar el IBAN o cambiar la cuenta bancaria",
    "solicitar duplicado de la póliza o certificado",
    "poner una queja o reclamación por mal servicio",
    "consulta irrelevante o fuera de contexto"
]

def obtener_umbral_dinamico(intent_detectado):
    intent = intent_detectado.lower()
    if any(p in intent for p in ["recibo", "pago", "iban", "cuenta"]):
        return 0.55 # Estricto para dinero
    if any(p in intent for p in ["accidente", "choque", "asistencia", "grúa", "agua"]):
        return 0.30 # Permisivo para urgencias
    return 0.45 # Estándar

# 5. BUCLE PRINCIPAL
print("\n" + "="*50)
print("🤖 SISTEMA SEGURPLUS v4.1 ACTIVADO")
print("="*50)

while True:
    usuario = input("\nTú: ")
    if usuario.lower() in ['salir', 'exit']: break
    if not usuario.strip(): continue

    # PASO A: Privacidad
    alerta_datos = analizar_seguridad(usuario)
    
    # PASO B: Clasificación (UNA SOLA VEZ)
    resultado = clasificador(usuario, candidate_labels=etiquetas)
    intent = resultado['labels'][0]
    confianza = resultado['scores'][0]
    
    # PASO C: Umbral Dinámico
    umbral_necesario = obtener_umbral_dinamico(intent)

    # PASO D: Lógica de Respuesta
    if intent == "consulta irrelevante o fuera de contexto":
        respuesta = "Lo siento, solo puedo ayudarte con gestiones relacionadas con tus seguros de coche, hogar o vida."
    
    elif confianza < umbral_necesario:
        respuesta = f"He detectado que podrías necesitar ayuda con algo relacionado con '{intent[:30]}...', pero no estoy seguro. ¿Me das más detalles?"
    
    else:
        # Usamos palabras clave para decidir la respuesta final
        if "accidente" in intent or "choque" in intent:
            respuesta = "Lamento el accidente. He activado el protocolo de siniestros de auto. ¿Necesitas una grúa?"
        elif "asistencia" in intent or "grúa" in intent:
            respuesta = "Entendido, estoy localizando la grúa más cercana a tu posición..."
        elif "agua" in intent or "goteras" in intent:
            respuesta = "Siento los daños por agua. ¿Ha afectado a algún vecino o es solo en tu vivienda?"
        elif "recibo" in intent or "pago" in intent or "iban" in intent:
            respuesta = "Accediendo a tu área de cliente para gestionar el pago o modificar la cuenta bancaria."
        elif "queja" in intent:
            respuesta = "Tomo nota de tu reclamación. Un gestor la revisará para darte una solución rápida."
        else:
            respuesta = f"Perfecto, procedo a gestionar tu solicitud sobre: {intent}."

    # SALIDA FINAL
    print(f"\nChatbot: {respuesta}")
    
    if alerta_datos:
        print("\n🛡️ [REPORTE DE PRIVACIDAD]")
        for dato in alerta_datos:
            print(f"  - ¡Cuidado! No es necesario que me des tu {dato['tipo']}. He anonimizado '{dato['anonimo']}' para protegerte.")

    print(f"  [Log: Intent detectado: '{intent[:40]}...' | Confianza: {confianza:.2f} | Umbral: {umbral_necesario}]")

2026-05-13 20:19:38.221659: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


⏳ Cargando cerebro de IA...


model.safetensors:   0%|          | 0.00/439M [00:00<?, ?B/s]